# Wang2025 Model Testing
Test the Wang2025 acoustic anomaly detection system with EAT backbone and ArcFace classification

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import torchaudio
import os
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel
from sklearn.metrics import roc_auc_score, precision_recall_curve
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 1. Label Encoding

In [ ]:
def _normalize_attr_value(v):
    """Normalize attribute values for label encoding."""
    if pd.isna(v):
        return ""
    s = str(v).strip()
    try:
        x = float(s.replace(",", "."))
        return str(int(x)) if x == int(x) else s
    except ValueError:
        return s

def _iter_machine_attribute_csvs(base_path):
    """Iterate over machine attribute CSVs."""
    if not os.path.isdir(base_path):
        return
    for name in sorted(os.listdir(base_path)):
        p = os.path.join(base_path, name)
        if os.path.isdir(p):
            ap = os.path.join(p, "attributes_00.csv")
            if os.path.isfile(ap):
                yield name, ap

def get_dcase_num_classes(base_path):
    """Get all unique classes from DCASE training data."""
    all_configs = []
    for machine_type, file in _iter_machine_attribute_csvs(base_path):
        df = pd.read_csv(file)
        train_df = df[df["file_name"].str.contains("train", na=False)].copy()
        
        for _, row in train_df.iterrows():
            domain = "source" if "source" in str(row["file_name"]) else "target"
            attr_cols = [col for col in train_df.columns if col.endswith("v")]
            attr_values = [_normalize_attr_value(row[col]) for col in attr_cols if pd.notna(row[col])]
            attr_values = [v for v in attr_values if v and v.lower() != "noattribute"]
            
            if not attr_values:
                config_label = f"{machine_type}_{domain}"
            else:
                config_label = f"{machine_type}_{domain}_{' '.join(attr_values)}"
            all_configs.append(config_label)
    
    unique_classes = sorted(list(set(all_configs)))
    print(f"Total Classes: {len(unique_classes)}")
    return unique_classes, len(unique_classes)

class DCASELabelEncoder:
    """Encode (machine, domain, attributes) tuples to class IDs."""
    def __init__(self, unique_configs):
        self.str_to_int = {label: i for i, label in enumerate(unique_configs)}
        self.int_to_str = {i: label for i, label in enumerate(unique_configs)}
        self.num_classes = len(unique_configs)

    def encode(self, machine, domain, attributes=None):
        machine = str(machine).strip()
        domain = str(domain).strip().lower()
        
        if not attributes or attributes == ["noAttribute"]:
            label_str = f"{machine}_{domain}"
        else:
            clean_attrs = [_normalize_attr_value(a) for a in attributes]
            clean_attrs = [a for a in clean_attrs if a and a.lower() != "noattribute"]
            label_str = f"{machine}_{domain}_{' '.join(clean_attrs)}"
        
        return self.str_to_int.get(label_str, -1)

## 2. Dataset and Data Loading

In [ ]:
def _resolve_train_wav_path(base_path, row):
    """Resolve WAV file path from CSV row."""
    fn = str(row["file_name"]).replace("\\\\", "/").replace("\\", "/")
    if "/" in fn:
        return os.path.normpath(os.path.join(base_path, *fn.split("/")))
    machine = row.get("_machine")
    if machine is None or (isinstance(machine, float) and pd.isna(machine)):
        raise ValueError(f"Invalid machine for: {fn}")
    stem = os.path.basename(fn)
    if not stem.lower().endswith(".wav"):
        stem = stem + ".wav"
    return os.path.normpath(os.path.join(base_path, str(machine), "train", stem))

class EATDCASEDataset(Dataset):
    """Load DCASE audio data with mel-spectrogram features and labels."""
    def __init__(self, df, encoder, base_path, target_length=1024):
        self.df = df
        self.encoder = encoder
        self.base_path = base_path
        self.target_length = target_length
        self.norm_mean = -4.268
        self.norm_std = 4.569

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = _resolve_train_wav_path(self.base_path, row)

        if not os.path.isfile(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")

        waveform, sr = torchaudio.load(file_path)
        if sr != 16000:
            waveform = torchaudio.transforms.Resample(sr, 16000)(waveform)

        waveform = waveform - waveform.mean()
        mel = torchaudio.compliance.kaldi.fbank(
            waveform, htk_compat=True, sample_frequency=16000,
            use_energy=False, window_type="hanning", num_mel_bins=128,
            dither=0.0, frame_shift=10,
        )

        n_frames = mel.shape[0]
        if n_frames < self.target_length:
            mel = F.pad(mel, (0, 0, 0, self.target_length - n_frames), "constant", 0)
        else:
            mel = mel[: self.target_length, :]

        mel = (mel - self.norm_mean) / (self.norm_std * 2)

        fn = str(row["file_name"]).replace("\\\\", "/").replace("\\", "/")
        machine = str(row.get("_machine", fn.split("/")[0])).strip()
        domain = "source" if "source" in fn else "target"
        attr_cols = [col for col in self.df.columns if col.endswith("v")]
        attr_values = [_normalize_attr_value(row[col]) for col in attr_cols if pd.notna(row[col])]
        attr_values = [v for v in attr_values if v and v.lower() != "noattribute"]

        target_id = self.encoder.encode(machine, domain, attr_values)
        return mel, torch.tensor(target_id).long()

def create_master_dataframe(base_path):
    """Create master training dataframe from all machine CSVs."""
    all_dfs = []
    for machine_type, csv_path in _iter_machine_attribute_csvs(base_path):
        df = pd.read_csv(csv_path)
        train_only = df[df["file_name"].str.contains("train", na=False)].copy()
        train_only["_machine"] = machine_type
        all_dfs.append(train_only)
    return pd.concat(all_dfs, ignore_index=True)

## 3. Model Architecture

In [ ]:
class ArcFaceLoss(nn.Module):
    """ArcFace loss for angular margin classification."""
    def __init__(self, in_features, out_features, s=30.0, m=0.50):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, input, label):
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        margin_theta = theta + self.m
        cos_margin = torch.cos(margin_theta)
        logits = self.s * cos_margin
        loss = F.cross_entropy(logits, label)
        return loss, logits

class EATBackbone(nn.Module):
    """EAT model wrapper with LoRA adapters."""
    def __init__(self, embed_dim=768, model_id="worstchan/EAT-base_epoch30_pretrain", apply_lora=False):
        super().__init__()
        self.embed_dim = embed_dim
        self.model = AutoModel.from_pretrained(model_id, trust_remote_code=True).eval()
        for p in self.model.parameters():
            p.requires_grad = False

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)
        with torch.no_grad():
            feat = self.model.extract_features(x)
        return feat

## 4. Testing Configuration

In [ ]:
# Configuration
BASE_PATH = r"data\dcase2025t2\dev_data\raw"
MODEL_ID = "worstchan/EAT-base_epoch30_pretrain"

# Get classes and encoder
labels_list, num_classes = get_dcase_num_classes(BASE_PATH)
encoder = DCASELabelEncoder(labels_list)
print(f"Encoder ready with {num_classes} classes")

## 5. Load and Test Data

In [ ]:
# Load master dataframe
master_df = create_master_dataframe(BASE_PATH)
print(f"Total training samples: {len(master_df)}")

# Create dataset and loader
dataset = EATDCASEDataset(master_df, encoder, BASE_PATH, target_length=1024)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

# Test loading
try:
    mel, label = dataset[0]
    print(f"✓ Sample loaded successfully")
    print(f"  Mel shape: {mel.shape}")
    print(f"  Label: {label.item()}")
except Exception as e:
    print(f"✗ Error loading sample: {e}")

## 6. Model Inference

In [ ]:
# Initialize model
backbone = EATBackbone(model_id=MODEL_ID).to(DEVICE)
classifier = ArcFaceLoss(768, num_classes).to(DEVICE)

# Test on a batch
mel_batch, label_batch = next(iter(loader))
mel_batch = mel_batch.to(DEVICE)
label_batch = label_batch.to(DEVICE)

with torch.no_grad():
    features = backbone(mel_batch)
    features_mean = features.mean(dim=1)
    loss, logits = classifier(features_mean, label_batch)

print(f"✓ Forward pass successful")
print(f"  Features shape: {features_mean.shape}")
print(f"  Logits shape: {logits.shape}")
print(f"  Loss: {loss.item():.4f}")